In [3]:
import pandas as pd
import numpy as np

master_timeseries_df = pd.read_csv("supply_chain_3yr_data.csv", encoding="ISO-8859-1")


In [18]:
master_timeseries_df.head()

,22197,84077,84879,85099B,85123A
Date,,,,,
2018-12-01,271.0,0.0,224.0,556.0,454.0
2018-12-02,74.0,384.0,503.0,48.0,309.0
2018-12-03,67.0,49.0,48.0,49.0,60.0
2018-12-04,0.0,0.0,0.0,0.0,0.0
2018-12-05,81.0,96.0,129.0,39.0,198.0


In [19]:
raw_df = pd.read_csv("Sales Transaction v.4a.csv", encoding="ISO-8859-1")


In [21]:
raw_df.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


In [27]:
import pandas as pd

print(f"Columns in your timeseries data: {clean_timeseries_df.columns.tolist()[:5]}")

top_5_products = [str(col).strip() for col in clean_timeseries_df.columns if col != 'Date'][:5]

raw_df_copy = raw_df.copy()
raw_df_copy['ProductNo'] = raw_df_copy['ProductNo'].astype(str).str.strip()

top_5_raw = raw_df_copy[raw_df_copy['ProductNo'].isin(top_5_products)]

product_insights = top_5_raw.groupby('ProductNo').agg(
    Product_Name=('ProductName', 'first'),
    Total_Quantity_Sold=('Quantity', 'sum'),
    Average_Order_Size=('Quantity', 'mean'),
    Max_Single_Order=('Quantity', 'max'),
    Total_Transactions=('TransactionNo', 'nunique'),
    Primary_Country=('Country', lambda x: x.mode()[0] if not x.empty else 'Unknown')
).reset_index()

product_insights = product_insights.sort_values(by='Total_Quantity_Sold', ascending=False)
product_insights['Average_Order_Size'] = product_insights['Average_Order_Size'].round(1)

print(" Top 5 Product Insights")
print(product_insights.to_string(index=False))

Columns in your timeseries data: ['Date', '22197', '84077', '84879', '85099B']

--- Top Product Insights ---
ProductNo                       Product_Name  Total_Quantity_Sold  Average_Order_Size  Max_Single_Order  Total_Transactions Primary_Country
    22197                     Popcorn Holder                56450                38.2              4300                1442  United Kingdom
    84077  World War 2 Gliders Asstd Designs                53847                99.3              4800                 540  United Kingdom
   85099B            Jumbo Bag Red Retrospot                47363                21.9              1200                2135  United Kingdom
    84879      Assorted Colour Bird Ornament                36445                24.3              2880                1467  United Kingdom
   85123A Cream Hanging Heart T-Light Holder                35378                14.9              1930                2311  United Kingdom


In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

if 'Date' in master_timeseries_df.columns:
    master_timeseries_df.set_index('Date', inplace=True)
    master_timeseries_df.index = pd.to_datetime(master_timeseries_df.index)

target_product = master_timeseries_df.columns[0]
df_eda = master_timeseries_df[[target_product]].copy()
df_eda.columns = ['Quantity']

df_eda['Quantity'] = pd.to_numeric(df_eda['Quantity'], errors='coerce').fillna(0)

df_eda['7-Day_MA'] = df_eda['Quantity'].rolling(window=7).mean()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df_eda.index, y=df_eda['Quantity'], mode='lines', name='Daily Sales', opacity=0.3))
fig1.add_trace(go.Scatter(x=df_eda.index, y=df_eda['7-Day_MA'], mode='lines', name='7-Day Moving Average', line=dict(color='red', width=2)))
fig1.update_layout(title=f'Daily Sales vs. 7-Day MA: Product {target_product}', template='plotly_white')
fig1.show()

window_size = 30
df_eda['Rolling_Mean'] = df_eda['Quantity'].rolling(window=window_size).mean()
df_eda['Rolling_Std'] = df_eda['Quantity'].rolling(window=window_size).std()

df_eda['Z_Score'] = np.where(df_eda['Rolling_Std'] == 0, 0,
                             (df_eda['Quantity'] - df_eda['Rolling_Mean']) / df_eda['Rolling_Std'])

anomalies = df_eda[df_eda['Z_Score'] > 3]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_eda.index, y=df_eda['Quantity'], mode='lines', name='Sales'))
fig2.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Quantity'], mode='markers', name='Anomaly (Z > 3)', marker=dict(color='red', size=8, symbol='x')))
fig2.update_layout(title=f'Real-Time Anomaly Detection (Rolling 30-Day Z-Score): Product {target_product}', template='plotly_white')
fig2.show()

In [10]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import plotly.graph_objects as go
import numpy as np
import pandas as pd

if 'Date' in master_timeseries_df.columns:
    master_timeseries_df.set_index('Date', inplace=True)
    master_timeseries_df.index = pd.to_datetime(master_timeseries_df.index)

target_product = master_timeseries_df.columns[0]
df_features = master_timeseries_df[[target_product]].copy()
df_features.columns = ['Target_Quantity']
df_features['Target_Quantity'] = pd.to_numeric(df_features['Target_Quantity'], errors='coerce').fillna(0)

prophet_df = df_features.reset_index()[['Date', 'Target_Quantity']]
prophet_df.columns = ['ds', 'y']

# Train/Test Split (Train: ~2 Years, Test: Year 3)
train_size = 730
train_df = prophet_df.iloc[:train_size]
test_df = prophet_df.iloc[train_size:]

# Initialize and Train Prophet
print("Training Prophet Model on Years 1 & 2...")
m = Prophet(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True)
m.fit(train_df)

future = m.make_future_dataframe(periods=len(test_df))
forecast = m.predict(future)

predictions = forecast['yhat'].tail(len(test_df)).values
predictions = np.maximum(0, predictions)
actuals = test_df['y'].values

mae_prophet = mean_absolute_error(actuals, predictions)
rmse_prophet = np.sqrt(mean_squared_error(actuals, predictions))

print(f"\n--- Prophet Model Results (Year 3 Test Set) ---")
print(f"Mean Absolute Error (MAE): {mae_prophet:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_prophet:.2f}")

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=test_df['ds'],
    y=actuals,
    mode='lines',
    name='Actual Sales (Year 3)',
    opacity=0.5,
    line=dict(color='blue')
))

fig4.add_trace(go.Scatter(
    x=test_df['ds'],
    y=predictions,
    mode='lines',
    name='Prophet Forecast',
    line=dict(color='red', dash='dash')
))

fig4.update_layout(
    title=f'Prophet Forecast vs Actuals: Product {target_product}',
    template='plotly_white'
)
fig4.show()

Training Prophet Model on Years 1 & 2...

--- Prophet Model Results (Year 3 Test Set) ---
Mean Absolute Error (MAE): 69.23
Root Mean Squared Error (RMSE): 101.87


In [11]:
import pandas as pd
import numpy as np
import holidays
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import plotly.graph_objects as go

target_product = master_timeseries_df.columns[0]
df_ml = master_timeseries_df[[target_product]].copy()
df_ml.columns = ['Target']
df_ml['Target'] = pd.to_numeric(df_ml['Target'], errors='coerce').fillna(0)

df_ml['DayOfWeek'] = df_ml.index.dayofweek
df_ml['Month'] = df_ml.index.month
df_ml['Is_Weekend'] = df_ml['DayOfWeek'].isin([5, 6]).astype(int)

uk_holidays = holidays.UK(years=[2019, 2020, 2021])
df_ml['Is_Holiday'] = df_ml.index.isin(uk_holidays).astype(int)

df_ml['Lag_1'] = df_ml['Target'].shift(1)
df_ml['Lag_7'] = df_ml['Target'].shift(7)
df_ml['Lag_30'] = df_ml['Target'].shift(30)

df_ml['Rolling_7D_Mean'] = df_ml['Target'].shift(1).rolling(window=7).mean()

df_ml = df_ml.dropna()

X = df_ml.drop(columns=['Target'])
y = df_ml['Target']

train_size = len(df_ml) - 365
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
test_dates = df_ml.index[train_size:]

# Linear Regression
print("Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = np.maximum(0, lr.predict(X_test)) # Force predictions to be >= 0
lr_mae = mean_absolute_error(y_test, lr_preds)
print(f"Linear Regression MAE: {lr_mae:.2f}")

# XGBoost
print("Training XGBoost...")
xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = np.maximum(0, xgb.predict(X_test))
xgb_mae = mean_absolute_error(y_test, xgb_preds)
print(f"XGBoost MAE: {xgb_mae:.2f}")

fig5 = go.Figure()

fig5.add_trace(go.Scatter(x=test_dates, y=y_test, mode='lines', name='Actual Sales (Year 3)', opacity=0.4, line=dict(color='blue')))

# Linear Regression
fig5.add_trace(go.Scatter(x=test_dates, y=lr_preds, mode='lines', name='Linear Regression', line=dict(color='orange', dash='dot')))

# XGBoost
fig5.add_trace(go.Scatter(x=test_dates, y=xgb_preds, mode='lines', name='XGBoost', line=dict(color='green', dash='dash')))

fig5.update_layout(
    title=f'Model Bake-Off: XGBoost vs Linear Regression (Product {target_product})',
    xaxis_title='Date',
    yaxis_title='Daily Quantity Sold',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig5.show()

/tmp/ipykernel_294/3083527681.py:22: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.



Training Linear Regression...
Linear Regression MAE: 76.93
Training XGBoost...
XGBoost MAE: 113.77


In [12]:
figC = go.Figure()
figC.add_trace(go.Scatter(x=test_dates, y=actuals, mode='lines', name='Actual Sales', opacity=0.3, line=dict(color='blue')))
figC.add_trace(go.Scatter(x=test_dates, y=xgb_preds, mode='lines', name='XGBoost Forecast', line=dict(color='green', dash='dash')))
figC.update_layout(title=f'Model Evaluation: XGBoost (MAE: {xgb_mae:.2f})', template='plotly_white')
figC.show()

In [14]:
figB = go.Figure()
figB.add_trace(go.Scatter(x=test_dates, y=actuals, mode='lines', name='Actual Sales', opacity=0.3, line=dict(color='blue')))
figB.add_trace(go.Scatter(x=test_dates, y=lr_preds, mode='lines', name='Linear Regression Forecasts', line=dict(color='red', dash='dash')))
figB.update_layout(title=f'Model Evaluation: Linear Regression (MAE: {lr_mae:.2f})', template='plotly_white')
figB.show()

In [60]:
import pandas as pd
import numpy as np
import holidays
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from prophet import Prophet
import plotly.graph_objects as go

target_product = master_timeseries_df.columns[0]
df_advanced = master_timeseries_df[[target_product]].copy()
df_advanced.columns = ['Quantity']
df_advanced['Quantity'] = pd.to_numeric(df_advanced['Quantity'], errors='coerce').fillna(0)

df_advanced['DayOfWeek'] = df_advanced.index.dayofweek
df_advanced['Is_Weekend'] = df_advanced['DayOfWeek'].isin([5, 6]).astype(int)
us_holidays = holidays.US(years=[2019, 2020, 2021])
df_advanced['Is_Holiday'] = df_advanced.index.isin(us_holidays).astype(int)

df_advanced['Lag_1'] = df_advanced['Quantity'].shift(1)
df_advanced['Lag_7'] = df_advanced['Quantity'].shift(7)
df_advanced['Rolling_7D_Mean'] = df_advanced['Quantity'].shift(1).rolling(window=7).mean()

df_advanced['Baseline_15D_Pred'] = df_advanced['Quantity'].shift(1).rolling(window=15).mean()

df_advanced = df_advanced.dropna()

split_idx = len(df_advanced) - 365
train_df = df_advanced.iloc[:split_idx]
test_df = df_advanced.iloc[split_idx:]
test_dates = test_df.index
actuals = test_df['Quantity'].values
test_length = len(test_df) # This handles the 374 vs 365 mismatch!

baseline_preds = test_df['Baseline_15D_Pred'].values

# Linear regression training
X_train = train_df.drop(columns=['Quantity', 'Baseline_15D_Pred'])
y_train = train_df['Quantity']
X_test = test_df.drop(columns=['Quantity', 'Baseline_15D_Pred'])

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = np.maximum(0, lr_model.predict(X_test))

#XGBoost training
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = np.maximum(0, xgb_model.predict(X_test))

#Prophet training
prophet_train_df = train_df.reset_index()[['Date', 'Quantity']]
prophet_train_df.columns = ['ds', 'y']
m_prophet = Prophet(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True)
m_prophet.fit(prophet_train_df)

future = m_prophet.make_future_dataframe(periods=test_length)
forecast = m_prophet.predict(future)
prophet_preds = np.maximum(0, forecast['yhat'].tail(test_length).values)

def calculate_wmape(actuals, forecasts):
    sum_abs_error = np.sum(np.abs(actuals - forecasts))
    sum_actuals = np.sum(actuals)
    return 0 if sum_actuals == 0 else (sum_abs_error / sum_actuals) * 100

def calculate_mape(actuals, forecasts):
    actuals, forecasts = np.array(actuals), np.array(forecasts)
    non_zero_mask = actuals != 0 # Prevent divide-by-zero crashes for 0 sale days
    if np.sum(non_zero_mask) == 0:
        return np.nan
    actuals_safe = actuals[non_zero_mask]
    forecasts_safe = forecasts[non_zero_mask]
    return np.mean(np.abs((actuals_safe - forecasts_safe) / actuals_safe)) * 100


baseline_mape = calculate_mape(actuals, baseline_preds)
lr_mape = calculate_mape(actuals, lr_preds)
prophet_mape = calculate_mape(actuals, prophet_preds)
xgb_mape = calculate_mape(actuals, xgb_preds)


print("--- MAPE ---")
print(f"Baseline (15-Day) MAPE: {baseline_mape:.2f}%")
print(f"Linear Regression MAPE: {lr_mape:.2f}%")
print(f"Prophet MAPE: {prophet_mape:.2f}%")
print(f"XGBoost MAPE: {xgb_mape:.2f}%")

--- Forecast Error Comparisons ---
Baseline (15-Day) -> WMAPE: 21.85%  |  MAPE: 24.88%
Linear Regression -> WMAPE: 20.81%  |  MAPE: 21.45%
Prophet           -> WMAPE: 19.07%  |  MAPE: 20.65%
XGBoost           -> WMAPE: 24.74%  |  MAPE: 25.15%
--- MAPE ---
Baseline (15-Day) MAPE: 24.88%
Linear Regression MAPE: 21.45%
Prophet MAPE: 20.65%
XGBoost MAPE: 25.15%


In [28]:
import plotly.express as px
import pandas as pd

if 'Date' in master_timeseries_df.columns:
    master_timeseries_df.set_index('Date', inplace=True)
    master_timeseries_df.index = pd.to_datetime(master_timeseries_df.index)

target_product = master_timeseries_df.columns[0]
df_weekly = master_timeseries_df[[target_product]].copy()
df_weekly.columns = ['Quantity']
df_weekly['Quantity'] = pd.to_numeric(df_weekly['Quantity'], errors='coerce').fillna(0)

df_weekly['DayOfWeek_Num'] = df_weekly.index.dayofweek
day_map = {0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'}
df_weekly['Day'] = df_weekly['DayOfWeek_Num'].map(day_map)

df_weekly['Day'] = pd.Categorical(df_weekly['Day'], categories=list(day_map.values()), ordered=True)

fig_box = px.box(df_weekly, x='Day', y='Quantity',
                 title=f'Demand Distribution by Day of Week: Product {target_product}',
                 color='Day', template='plotly_white')
fig_box.update_layout(showlegend=False, yaxis_title='Daily Units Sold')
fig_box.show()

In [43]:
import plotly.express as px
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Creates a lookup dictionary: { '22197': 'Popcorn Holder', ... }
mapping_df = raw_df[['ProductNo', 'ProductName']].dropna().drop_duplicates(subset=['ProductNo'])
mapping_df['ProductNo'] = mapping_df['ProductNo'].astype(str).str.strip()
product_dict = dict(zip(mapping_df['ProductNo'], mapping_df['ProductName']))

if 'Date' in master_timeseries_df.columns:
    master_timeseries_df.set_index('Date', inplace=True)
master_timeseries_df.index = pd.to_datetime(master_timeseries_df.index)

target_id = str(master_timeseries_df.columns[0]).strip()
product_name = product_dict.get(target_id, f"Product {target_id}")

df_weekly = master_timeseries_df[[target_id]].copy()
df_weekly.columns = ['Quantity']
df_weekly['Quantity'] = pd.to_numeric(df_weekly['Quantity'], errors='coerce').fillna(0)

df_weekly['DayOfWeek_Num'] = df_weekly.index.dayofweek
day_map = {0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'}
df_weekly['Day'] = df_weekly['DayOfWeek_Num'].map(day_map)
df_weekly['Day'] = pd.Categorical(df_weekly['Day'], categories=list(day_map.values()), ordered=True)

fig_box = px.box(df_weekly, x='Day', y='Quantity',
                 title=f'Weekly Demand Distribution: {product_name}',
                 color='Day', template='plotly_white')

fig_box.update_layout(
    showlegend=False,
    yaxis_title='Daily Units Sold',
    xaxis_title='Day of the Week'
)
fig_box.show()

In [33]:
import plotly.express as px
import pandas as pd

correlation_matrix = master_timeseries_df.corr()

mapping_df = raw_df[['ProductNo', 'ProductName']].dropna().drop_duplicates(subset=['ProductNo'])
mapping_df['ProductNo'] = mapping_df['ProductNo'].astype(str).str.strip()
product_dict = dict(zip(mapping_df['ProductNo'], mapping_df['ProductName']))

correlation_matrix.index = correlation_matrix.index.astype(str).str.strip()
correlation_matrix.columns = correlation_matrix.columns.astype(str).str.strip()

correlation_matrix_named = correlation_matrix.rename(index=product_dict, columns=product_dict)

fig_heat = px.imshow(correlation_matrix_named,
                     text_auto='.2f',
                     aspect="auto",
                     color_continuous_scale='Blues',
                     title='Demand Correlation Matrix: Top 5 Products')

fig_heat.update_layout(
    template='plotly_white',
    xaxis_tickangle=-45,
    margin=dict(l=150, b=100)
)
fig_heat.show()

In [36]:
df = pd.read_csv('Sales Transaction v.4a.csv')
df_mba = df[df['Country'] == 'United Kingdom']

In [37]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df_mba = raw_df.dropna(subset=['TransactionNo', 'ProductName']).copy()
df_mba['TransactionNo'] = df_mba['TransactionNo'].astype(str)
df_mba['ProductName'] = df_mba['ProductName'].str.strip() # Clean up text

print("Building the basket matrix (this might take a minute)...")
basket = (df_mba.groupby(['TransactionNo', 'ProductName'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('TransactionNo'))

def encode_units(x):
    if x <= 0: return 0
    if x >= 1: return 1

basket_sets = basket.applymap(encode_units)

print("Finding frequent itemsets...")
frequent_itemsets = apriori(basket_sets, min_support=0.02, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

top_rules = rules.sort_values('lift', ascending=False).head(10)
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: list(x)[0])
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: list(x)[0])

print("\n--- Top 10 Products Frequently Bought Together ---")
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

Building the basket matrix (this might take a minute)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Finding frequent itemsets...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v


--- Top 10 Products Frequently Bought Together ---
                      antecedents                       consequents  support  confidence      lift
  Green Regency Teacup And Saucer    Pink Regency Teacup And Saucer 0.023358    0.705729 21.378250
   Pink Regency Teacup And Saucer   Green Regency Teacup And Saucer 0.023358    0.707572 21.378250
  Roses Regency Teacup And Saucer   Green Regency Teacup And Saucer 0.023358    0.904841 20.685655
  Green Regency Teacup And Saucer   Roses Regency Teacup And Saucer 0.023358    0.533990 20.685655
   Pink Regency Teacup And Saucer   Green Regency Teacup And Saucer 0.027280    0.826371 18.891731
  Green Regency Teacup And Saucer    Pink Regency Teacup And Saucer 0.027280    0.623645 18.891731
  Green Regency Teacup And Saucer   Roses Regency Teacup And Saucer 0.023358    0.856240 18.638082
  Roses Regency Teacup And Saucer   Green Regency Teacup And Saucer 0.023358    0.508443 18.638082
 Gardeners Kneeling Pad Keep Calm Gardeners Kneeling Pad 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

In [39]:
import plotly.express as px

df_seasonal = master_timeseries_df.copy()
df_seasonal['DayOfYear'] = df_seasonal.index.dayofyear
seasonal_profile = df_seasonal.groupby('DayOfYear').mean()

seasonal_norm = (seasonal_profile - seasonal_profile.min()) / (seasonal_profile.max() - seasonal_profile.min())

fig_fingerprint = px.line(seasonal_norm,
                          title='Seasonal Fingerprint: How Top 5 Products Synchronize Over a Year',
                          labels={'value': 'Normalized Demand', 'DayOfYear': 'Day of Year (1-365)'},
                          template='plotly_white')
fig_fingerprint.show()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

In [42]:
import pandas as pd
import plotly.graph_objects as go

import warnings
import logging

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

mapping_df = raw_df[['ProductNo', 'ProductName']].dropna().drop_duplicates(subset=['ProductNo'])
mapping_df['ProductNo'] = mapping_df['ProductNo'].astype(str).str.strip()
product_dict = dict(zip(mapping_df['ProductNo'], mapping_df['ProductName']))

top_5_ids = [str(col).strip() for col in master_timeseries_df.columns if col != 'Date'][:5]

for product_id in top_5_ids:
    product_name = product_dict.get(product_id, f"Product {product_id}")

    df_single = master_timeseries_df[[product_id]].copy()
    df_single.index = pd.to_datetime(df_single.index)

    # Aggregate by Month to find the 'Average Seasonal Cycle'
    # We take the mean across all 3 years for each month
    monthly_stats = df_single.resample('M').mean()
    monthly_stats['Month_Name'] = monthly_stats.index.strftime('%b')

    # Sort them in calendar order
    month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    seasonal_cycle = monthly_stats.groupby('Month_Name').mean().reindex(month_order)

    # Generate the Plot
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=seasonal_cycle.index,
        y=seasonal_cycle[product_id],
        marker_color='indigo',
        opacity=0.75,
        name='Avg Demand'
    ))

    fig.update_layout(
        title=f'Annual Seasonality Profile: {product_name}',
        xaxis_title='Month',
        yaxis_title='Average Daily Units Sold',
        template='plotly_white',
        height=400
    )

    fig.show()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

In [44]:
import pandas as pd
import numpy as np
import plotly.express as px

window_size = 30
df_heat = master_timeseries_df[[target_product]].copy()
df_heat.columns = ['Quantity']

df_heat['Rolling_Mean'] = df_heat['Quantity'].rolling(window=window_size).mean()
df_heat['Rolling_Std'] = df_heat['Quantity'].rolling(window=window_size).std()

df_heat['Z_Score'] = np.where(df_heat['Rolling_Std'] == 0, 0,
                             (df_heat['Quantity'] - df_heat['Rolling_Mean']) / df_heat['Rolling_Std'])

df_heat['Is_Anomaly'] = (df_heat['Z_Score'] > 3).astype(int)
df_heat['Month'] = df_heat.index.strftime('%B')
df_heat['DayOfWeek'] = df_heat.index.strftime('%A')

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heatmap_data = df_heat.groupby(['Month', 'DayOfWeek'])['Is_Anomaly'].sum().reset_index()
heatmap_pivot = heatmap_data.pivot(index='DayOfWeek', columns='Month', values='Is_Anomaly')
heatmap_pivot = heatmap_pivot.reindex(index=day_order, columns=month_order).fillna(0)

fig_heat = px.imshow(heatmap_pivot,
                     labels=dict(x="Month", y="Day of Week", color="Anomaly Count"),
                     x=month_order,
                     y=day_order,
                     color_continuous_scale='Reds',
                     title=f'Anomaly Frequency Heatmap: {product_name}')

fig_heat.update_layout(template='plotly_white')
fig_heat.show()

In [46]:
import pandas as pd
import numpy as np
import plotly.express as px

mapping_df = raw_df[['ProductNo', 'ProductName']].dropna().drop_duplicates(subset=['ProductNo'])
mapping_df['ProductNo'] = mapping_df['ProductNo'].astype(str).str.strip()
product_dict = dict(zip(mapping_df['ProductNo'], mapping_df['ProductName']))

anomaly_counts = {}
window_size = 30

for product_id in master_timeseries_df.columns:
    if product_id == 'Date': continue

    col = master_timeseries_df[product_id]
    rolling_mean = col.rolling(window=window_size).mean()
    rolling_std = col.rolling(window=window_size).std()

    z_scores = np.where(rolling_std == 0, 0, (col - rolling_mean) / rolling_std)
    total_anomalies = np.sum(z_scores > 3)

    p_name = product_dict.get(str(product_id).strip(), f"ID: {product_id}")
    anomaly_counts[p_name] = total_anomalies

df_compare = pd.DataFrame(list(anomaly_counts.items()), columns=['Product', 'Anomaly_Count'])
fig_compare = px.bar(df_compare, x='Product', y='Anomaly_Count',
                     title='Total Anomalies Identified (Z > 3) by Product',
                     color='Anomaly_Count', color_continuous_scale='Reds')
fig_compare.update_layout(template='plotly_white', xaxis_tickangle=-45)
fig_compare.show()

In [47]:
import plotly.graph_objects as go

df_eda['Upper_Bound'] = df_eda['Rolling_Mean'] + (3 * df_eda['Rolling_Std'])
df_eda['Lower_Bound'] = df_eda['Rolling_Mean'] - (3 * df_eda['Rolling_Std'])

fig_logic = go.Figure()

fig_logic.add_trace(go.Scatter(
    x=df_eda.index.tolist() + df_eda.index.tolist()[::-1],
    y=df_eda['Upper_Bound'].tolist() + df_eda['Lower_Bound'].fillna(0).tolist()[::-1],
    fill='toself', fillcolor='rgba(173, 216, 230, 0.2)',
    line=dict(color='rgba(255,255,255,0)'), name='Normal Range (3σ)'
))

fig_logic.add_trace(go.Scatter(x=df_eda.index, y=df_eda['Quantity'], name='Daily Sales', line=dict(color='black', width=1)))
fig_logic.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Quantity'], mode='markers', name='Detected Anomaly', marker=dict(color='red', size=7, symbol='x')))

fig_logic.update_layout(title=f'Anomaly Logic: Dynamic 3-Sigma Thresholding ({product_name})', template='plotly_white')
fig_logic.show()

In [55]:
import numpy as np
import pandas as pd

def calculate_wmape(y_true, y_pred):
    """Calculates Weighted Mean Absolute Percentage Error"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

window_size = 15

target_product = master_timeseries_df.columns[0]

df_single = master_timeseries_df[[target_product]].copy()
df_single.columns = ['Quantity']
df_single['Baseline_Forecast'] = df_single['Quantity'].rolling(window=window_size).mean().shift(1)

df_eval = df_single.dropna(subset=['Baseline_Forecast'])

actuals = df_eval['Quantity']
predictions = df_eval['Baseline_Forecast']

baseline_wmape = calculate_wmape(actuals, predictions)

print(f"--- Baseline Model Evaluation ---")
print(f"Sliding Window Size : {window_size} days")
print(f"Baseline WMAPE      : {baseline_wmape:.4f} ({baseline_wmape * 100:.2f}%)")


--- Baseline Model Evaluation ---
Sliding Window Size : 15 days
Baseline WMAPE      : 0.3538 (35.38%)
